# Query service — notebook interface

Run `uv run query-service build` once first, then `uv run jupyter lab` (or open this file in VS Code with the project venv selected).

Everything here goes through the same guards as the CLI: values bound, identifiers allowlisted, one read statement, read-only connection, capped rows.

In [ ]:
from queryservice import QueryService, UnsafeQuery

service = QueryService("../data/warehouse.duckdb", cache_dir="../.cache/queryservice")
for query in service.catalog():
    print(query.name, "--", query.description)

## A catalog query

`QueryResult` renders as a table with its provenance: row count, whether it came from cache, and how long it took.

In [ ]:
result = service.run(
    "top_customers",
    region="EUROPE",
    start_date="1995-01-01",
    end_date="1996-01-01",
    min_orders=5,
    limit=10,
)
result

In [ ]:
# Second call is served from the cache -- same frame, no warehouse round trip.
again = service.run(
    "top_customers",
    region="EUROPE",
    start_date="1995-01-01",
    end_date="1996-01-01",
    min_orders=5,
    limit=10,
)
print(again.cached, round(result.elapsed_ms, 1), "ms ->", round(again.elapsed_ms, 1), "ms")

## The dimension is an allowlisted identifier

A column name cannot be a bound parameter, so `revenue_by_dimension` declares the set it will accept and anything else is refused before the SQL is built.

In [ ]:
window = {"start_date": "1995-01-01", "end_date": "1996-01-01"}
service.run("revenue_by_dimension", dimension="r_name", **window).frame

In [ ]:
try:
    service.run("revenue_by_dimension", dimension="n_name; DROP TABLE customer", **window)
except UnsafeQuery as exc:
    print("rejected:", exc)

## Ad-hoc SQL is allowed, but not unguarded

In [ ]:
service.run_sql(
    "SELECT n_name, count(*) AS customers FROM customer"
    " JOIN nation ON n_nationkey = c_nationkey"
    " WHERE c_mktsegment = $segment GROUP BY 1 ORDER BY 2 DESC",
    {"segment": "MACHINERY"},
    limit=5,
).frame

In [ ]:
for payload in [
    "DROP TABLE customer",
    "SELECT 1; DELETE FROM orders",
    "PRAGMA database_list",
]:
    try:
        service.run_sql(payload)
    except UnsafeQuery as exc:
        print(f"{payload!r:<35} -> {exc}")

In [ ]:
service.close()